In [2]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
import json

from pdfminer.high_level import extract_text

from app.services.llm.open_ai_service_provider import OpenAIServiceProvider
from app.services.llm.prompts.user_analysis import (
    QUERY_ANALYSIS_PROMPT,
    RESUME_ANALYSIS_PROMPT,
)

from app.services.storage.db_controller import DBController
from app.services.storage.utils import session_scope
from sqlalchemy import create_engine

In [3]:
user_input = "我2026年2月 master 毕业，想在中国找一些跟数据科学、机器学习相关的岗位"
user_resume_file_path = "test_resume_cn.pdf"

In [4]:
llm_service_provider = OpenAIServiceProvider(
    api_url="https://api.deepseek.com", api_key="sk-135fe459060d4443ab30b8ae1f68f900"
)

# handle the user query
prompt = QUERY_ANALYSIS_PROMPT
messages = [
    {
        "role": "system",
        "content": prompt.format(curr_date=datetime.today().strftime("%Y-%m-%d")),
    },
    {"role": "user", "content": user_input},
]
str_res = llm_service_provider.get_completion(
    model_name="deepseek-chat",
    messages=messages,
    other_prompt_args={"response_format": {"type": "json_object"}},
)

user_analysis_res = json.loads(str_res)


# handle the resume analysis
prompt = RESUME_ANALYSIS_PROMPT
resume_text = extract_text(user_resume_file_path)
messages = [
    {"role": "system", "content": prompt},
    {"role": "user", "content": resume_text},
]
str_res = llm_service_provider.get_completion(
    model_name="deepseek-chat",
    messages=messages,
    other_prompt_args={"response_format": {"type": "json_object"}},
)

resume_analysis_res = json.loads(str_res)

In [5]:
user_analysis_res

{'intended_company': [],
 'intended_location': ['中国'],
 'intended_company_type': ['国企', '私企', '外企'],
 'intended_industry': [],
 'intended_position': ['数据科学', '机器学习'],
 'job_type': ['实习']}

In [6]:
resume_analysis_res

{'education': [{'school': '新加坡国立大学',
   'degree': '硕士',
   'major': '计算机科学（人工智能方向）',
   'graduation_year': 2026},
  {'school': '香港科技大学',
   'degree': '本科',
   'major': '计量金融与计算机科学',
   'graduation_year': 2024}],
 'work_experience': [{'company': 'Intact Financial (HK) Limited',
   'position': '数据科学实习生',
   'responsibilities': ['开发智能文档解析软件',
    '设计并实现保险文件的聚类与主题建模算法',
    '开发针对法语文件的文档解析和数据加工程序',
    '开发自动生成仿真保险文件数据的程序']},
  {'company': '中国电信（北京研究院）',
   'position': '算法实习生',
   'responsibilities': ['研究法律文本的知识表征学习',
    '微调大语言模型进行摘要生成',
    '开发获取训练数据的Python程序',
    '管理并协调多个数据标注项目']}],
 'skills': ['Python',
  'SQL',
  'Java',
  'C++',
  'PyTorch',
  'NumPy',
  'Pandas',
  'Scikit-learn',
  'OpenCV',
  'PowerBI',
  'Matplotlib',
  'Seaborn',
  'AWS S3',
  'Spark',
  'Databricks',
  'MongoDB',
  'Git',
  'GitHub',
  'Data Version Control (DVC)']}

In [7]:
from app.models.base import Base
from app.models.user import User, Resume, UserQueryPreference
import uuid

In [8]:
engine = create_engine("sqlite:///test.db")

In [10]:
mt = Base.metadata

mt.create_all(bind=engine)

In [11]:
db_controller = DBController(engine=engine)


from app.models.user import User, Resume, UserQueryPreference

In [22]:
# class UserQueryPreference(Base):
#     __tablename__ = 'user_query_db'

#     id = Column(Uuid, primary_key=True)

#     user_id = Column(Uuid, ForeignKey("user_db.id"))
#     intended_company = Column(JSON, default=[])
#     intended_company_type = Column(JSON, default=[])
#     intended_location = Column(JSON, default=[])
#     intended_industry = Column(JSON, default=[])
#     intended_position = Column(JSON, default=[])
#     job_type = Column(JSON, default=[])

In [12]:
user_query_perference = UserQueryPreference(
    id=uuid.uuid1(),
    user_id=uuid.uuid1(),
    intended_company=user_analysis_res["intended_company"],
    intended_company_type=user_analysis_res["intended_company_type"],
    intended_industry=user_analysis_res["intended_industry"],
    intended_position=user_analysis_res["intended_position"],
    job_type=user_analysis_res["job_type"],
)


with session_scope(db_controller.session_maker) as session:
    db_controller.insert_user_query_pref(session, user_query_perference)